# Task 1

In [ ]:

metin = "Bir çığ düşüşünü gördüm."

for harf in metin:
    unicode_kodu = f"U+{ord(harf):04X}"
    utf8 = harf.encode("utf-8")

    print(
        f"{repr(harf):5} | "
        f"Unicode: {unicode_kodu} | "
        f"Bytes: {list(utf8)} | "
        f"Boyut: {len(utf8)}"
    )

print("\nKarakter sayısı:", len(metin))
print("Toplam byte:", len(metin.encode("utf-8")))


'B'   | Unicode: U+0042 | Bytes: [66] | Boyut: 1
'i'   | Unicode: U+0069 | Bytes: [105] | Boyut: 1
'r'   | Unicode: U+0072 | Bytes: [114] | Boyut: 1
' '   | Unicode: U+0020 | Bytes: [32] | Boyut: 1
'ç'   | Unicode: U+00E7 | Bytes: [195, 167] | Boyut: 2
'ı'   | Unicode: U+0131 | Bytes: [196, 177] | Boyut: 2
'ğ'   | Unicode: U+011F | Bytes: [196, 159] | Boyut: 2
' '   | Unicode: U+0020 | Bytes: [32] | Boyut: 1
'd'   | Unicode: U+0064 | Bytes: [100] | Boyut: 1
'ü'   | Unicode: U+00FC | Bytes: [195, 188] | Boyut: 2
'ş'   | Unicode: U+015F | Bytes: [197, 159] | Boyut: 2
'ü'   | Unicode: U+00FC | Bytes: [195, 188] | Boyut: 2
'ş'   | Unicode: U+015F | Bytes: [197, 159] | Boyut: 2
'ü'   | Unicode: U+00FC | Bytes: [195, 188] | Boyut: 2
'n'   | Unicode: U+006E | Bytes: [110] | Boyut: 1
'ü'   | Unicode: U+00FC | Bytes: [195, 188] | Boyut: 2
' '   | Unicode: U+0020 | Bytes: [32] | Boyut: 1
'g'   | Unicode: U+0067 | Bytes: [103] | Boyut: 1
'ö'   | Unicode: U+00F6 | Bytes: [195, 182] | Boyut: 2
'r' 

# **Task 2**

In [ ]:
import argparse
import os
import sys
import time


TEXT_PATH = "turkish (1).txt"
VOCAB_SIZE = 276
SHOW = 20                   # kaç birleştirme yazdırılsın


# --------------------------------------------------------------------------
# 1) Ardışık ikilileri say        (minbpe/base.py -> get_stats)
# --------------------------------------------------------------------------
def get_stats(ids):
    """[1, 2, 3, 1, 2] -> {(1, 2): 2, (2, 3): 1, (3, 1): 1}"""
    counts = {}
    for pair in zip(ids, ids[1:]):          # ardışık elemanlar
        counts[pair] = counts.get(pair, 0) + 1
    return counts


# --------------------------------------------------------------------------
# 2) İkiliyi yeni token'la değiştir  (minbpe/base.py -> merge)
# --------------------------------------------------------------------------
def merge(ids, pair, idx):
    """ids=[1, 2, 3, 1, 2], pair=(1, 2), idx=4  ->  [4, 3, 4]"""
    newids = []
    i = 0
    while i < len(ids):
        # son pozisyonda değilsek VE ikili eşleşiyorsa değiştir
        if ids[i] == pair[0] and i < len(ids) - 1 and ids[i + 1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids


# --------------------------------------------------------------------------
# 3) Eğitim döngüsü                 (minbpe/basic.py -> BasicTokenizer.train)
# --------------------------------------------------------------------------
def train(text, vocab_size, log_every=25):
    assert vocab_size >= 256
    num_merges = vocab_size - 256

    text_bytes = text.encode("utf-8")       # ham byte'lar
    ids = list(text_bytes)                  # 0..255 arası tamsayılar

    merges = {}                                     # (int, int) -> int
    vocab = {idx: bytes([idx]) for idx in range(256)}  # int -> bytes
    history = []                                    # (pair, idx, frekans)

    t0 = time.time()
    for i in range(num_merges):
        stats = get_stats(ids)                  # her ikilinin kaç kez geçtiği
        pair = max(stats, key=stats.get)        # en sık ikili
        idx = 256 + i                           # yeni token id'si
        ids = merge(ids, pair, idx)             # tüm geçişleri değiştir
        merges[pair] = idx
        vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
        history.append((pair, idx, stats[pair]))

        if log_every and (i + 1) % log_every == 0 and num_merges > log_every:
            el = time.time() - t0
            eta = el / (i + 1) * (num_merges - i - 1)
            print(f"  [{i + 1:>4}/{num_merges}]  ids: {len(ids):,}  "
                  f"| geçen {el:5.1f}s | kalan ~{eta:5.1f}s")

    return list(text_bytes), ids, merges, vocab, history


# --------------------------------------------------------------------------
# Token'ı okunur metne çevirme
# --------------------------------------------------------------------------
def is_valid_utf8(b):
    try:
        b.decode("utf-8")
        return True
    except UnicodeDecodeError:
        return False


def show(b):
    """Byte dizisini metin olarak göster.
    Tek başına harf olmayan UTF-8 parçaları  \\xc4  şeklinde, boşluk ␣, satır sonu ⏎."""
    s = b.decode("utf-8", errors="backslashreplace")
    return s.replace(" ", "␣").replace("\n", "⏎").replace("\r", "␍").replace("\t", "⇥")


def kind(pair, vocab):
    left, right = vocab[pair[0]], vocab[pair[1]]
    merged = left + right
    if not (is_valid_utf8(left) and is_valid_utf8(right)):
        return "utf8-harf" if is_valid_utf8(merged) else "utf8-parça"
    if b" " in merged or b"\n" in merged:
        return "boşluk"
    return "harf grubu"


KIND_TR = {
    "utf8-harf":  "2 byte birleşip Türkçe harf oldu",
    "utf8-parça": "yarım UTF-8 parçası",
    "boşluk":     "boşluk/satır sonu içeriyor (kelime sınırı)",
    "harf grubu": "harf grubu",
}


# --------------------------------------------------------------------------
# main
# --------------------------------------------------------------------------
def running_in_notebook_cell():

    launcher = os.path.basename(sys.argv[0])
    kernel_json = "-f" in sys.argv and any(a.endswith(".json") for a in sys.argv[1:])
    return "kernel" in launcher or kernel_json


def get_args():
    # Notebook hücresinde argv'yi okumuyoruz; yukarıdaki sabitler kullanılıyor.
    if running_in_notebook_cell():
        return argparse.Namespace(path=TEXT_PATH, vocab=VOCAB_SIZE, show=SHOW)

    ap = argparse.ArgumentParser()
    ap.add_argument("path", nargs="?", default=TEXT_PATH, help="Türkçe metin dosyası")
    ap.add_argument("--vocab", type=int, default=VOCAB_SIZE,
                    help="hedef vocab boyutu (videoda 276 = 20 birleştirme)")
    ap.add_argument("--show", type=int, default=SHOW, help="kaç birleştirme yazdırılsın")
    return ap.parse_args()


def main():
    args = get_args()

    if not os.path.exists(args.path):
        txts = [f for f in os.listdir(".") if f.endswith(".txt")]
        print(f"Dosya bulunamadı: {args.path!r}")
        print(f"Bu klasördeki .txt dosyaları: {txts}")
        print("TEXT_PATH'i bunlardan birine ayarla.")
        return

    text = open(args.path, "r", encoding="utf-8").read()

    print("=" * 78)
    print("METİN")
    print("=" * 78)
    n_bytes = len(text.encode("utf-8"))
    non_ascii = sum(1 for ch in text if ord(ch) > 127)
    print(f"Okunan dosya        : {os.path.abspath(args.path)}")
    print(f"Karakter sayısı     : {len(text):,}")
    print(f"UTF-8 byte sayısı   : {n_bytes:,}")
    print(f"Byte / karakter     : {n_bytes / len(text):.3f}")
    print(f"ASCII dışı karakter : {non_ascii:,}  (%{100 * non_ascii / len(text):.1f})")
    if non_ascii == 0:
        print("!! UYARI: Metinde hiç Türkçe karakter yok, yanlış dosya okunmuş olabilir. "
              "TEXT_PATH'i kontrol et.")
    print("Türkçe harfler UTF-8'de 2'şer byte:")
    print("   " + "   ".join(f"{c}={tuple(c.encode('utf-8'))}" for c in "çğıöşü"))

    print()
    print("=" * 78)
    print(f"EĞİTİM  (vocab 256 -> {args.vocab}, {args.vocab - 256} birleştirme)")
    print("=" * 78)
    tokens, ids, merges, vocab, history = train(text, args.vocab)

    # ---- İlk N birleştirme (metin olarak) ----
    n_show = min(args.show, len(history))
    print()
    print("=" * 78)
    print(f"İLK {n_show} BİRLEŞTİRME")
    print("=" * 78)
    print(f"{'#':>3}  {'ikili (id)':<12} {'yeni':>4}   {'sol':>6} + {'sağ':<6} -> {'token':<10} {'frekans':>8}  tür")
    print("-" * 78)
    for r, (pair, idx, count) in enumerate(history[:n_show], 1):
        l, rgt, new = show(vocab[pair[0]]), show(vocab[pair[1]]), show(vocab[idx])
        print(f"{r:>3}  {str(pair):<12} {idx:>4}   {l:>6} + {rgt:<6} -> {new:<10} {count:>8,}  "
              f"{KIND_TR[kind(pair, vocab)]}")

    # türlere göre özet
    counts = {}
    for pair, _, _ in history[:n_show]:
        k = kind(pair, vocab)
        counts[k] = counts.get(k, 0) + 1
    print("-" * 78)
    print("Özet: " + " | ".join(f"{KIND_TR[k]}: {v}" for k, v in counts.items()))

    # ---- Sıkıştırma oranı (videodaki gibi) ----
    print()
    print("=" * 78)
    print("SIKIŞTIRMA")
    print("=" * 78)
    print("tokens length:", len(tokens))
    print("ids length:", len(ids))
    print(f"compression ratio: {len(tokens) / len(ids):.2f}X   (byte sayısı / token sayısı)")


if __name__ == "__main__":
    main()

Dosya bulunamadı: 'turkish (1).txt'
Bu klasördeki .txt dosyaları: []
TEXT_PATH'i bunlardan birine ayarla.


# **Task 3**

In [ ]:


def get_stats(ids):
    """Ardışık çiftleri say. Örnek: [1, 2, 3, 1, 2] -> {(1, 2): 2, (2, 3): 1, (3, 1): 1}"""
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts


def merge(ids, pair, idx):
    """ids içinde 'pair' çiftinin geçtiği her yeri yeni token 'idx' ile değiştir.
    Örnek: ids=[1, 2, 3, 1, 2], pair=(1, 2), idx=4 -> [4, 3, 4]"""
    newids = []
    i = 0
    while i < len(ids):
        # son pozisyonda değilsek VE çift eşleşiyorsa -> birleştir
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids


def train(text, vocab_size, verbose=False):
    """Byte-level BPE eğitimi. Dönen değer: merges sözlüğü {(int, int): int}"""
    assert vocab_size >= 256
    num_merges = vocab_size - 256

    ids = list(text.encode("utf-8"))  # 0..255 arası tamsayılar
    merges = {}
    for i in range(num_merges):
        stats = get_stats(ids)
        if not stats:  # birleştirilecek çift kalmadı
            break
        pair = max(stats, key=stats.get)  # en sık geçen çift
        idx = 256 + i                     # yeni token ID'si
        ids = merge(ids, pair, idx)
        merges[pair] = idx
        if verbose:
            print(f"merge {i + 1}/{num_merges}: {pair} -> {idx}  ({stats[pair]} kez)")
    return merges


# Eğitim metni (bilerek emoji İÇERMİYOR: emojinin yine de doğru encode/decode
# edildiğini 4. bölümde göreceğiz — çünkü 256 byte'ın hepsi zaten vocab'ta).
training_text = """
Sabah erkenden yağmur yağdı. Yağmur dinince güneş açtı ve sokaklar ışıl ışıl oldu.
Dağların tepesinde kar vardı; çığ tehlikesi yüzünden yollar kapatıldı. Köylüler
çığ uyarısını duyunca evlerine döndü. Öğleden sonra yine güneş açtı, çocuklar
bahçede oynadı, kediler güneşte uyudu. Akşam olunca rüzgâr çıktı, ağaçlar eğildi,
yapraklar savruldu. Gece yağmur yine yağdı, sabah yine güneş doğdu.
Türkçe metinlerde ç, ğ, ı, ö, ş, ü harfleri sık geçer; bu harfler UTF-8'de iki
byte ile yazılır. Tokenizer bu byte çiftlerini sık gördükçe birleştirmeyi öğrenir.
Çığ, dağ, yağ, sağ, bağ, ağ kelimelerinde ğ harfi var; güneş, kış, taş, baş, iş
kelimelerinde ş harfi var. Işık, ılık, ıslak kelimeleri ı ile başlar.
Yağmur yağdı, güneş açtı, çığ düştü, kar eridi, ırmak taştı, dağ yeşerdi.
"""

VOCAB_SIZE = 300  # 256 ham byte + 44 merge
merges = train(training_text, VOCAB_SIZE)

vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

print("=" * 70)
print("=" * 70)
for (p0, p1), idx in list(merges.items())[:15]:
    # bazı token'lar yarım bir UTF-8 karakteri olabilir -> replace ile göster
    print(f"{(p0, p1)} -> {idx}  {vocab[idx].decode('utf-8', errors='replace')!r}")


# %% ---------------------------------------------------------------------------
# 1. ENCODE — metin -> token ID listesi
# -----------------------------------------------------------------------------
def encode(text):


    tokens = list(text.encode("utf-8"))

    # len < 2 ise birleştirilecek çift yok (boş string / tek byte'lık metin).
    # Bu kontrol olmazsa get_stats boş döner ve min() hata verir.
    while len(tokens) >= 2:
        stats = get_stats(tokens)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break
        idx = merges[pair]
        tokens = merge(tokens, pair, idx)
    return tokens


# %% ---------------------------------------------------------------------------
# 2. DECODE — token ID listesi -> metin
# -----------------------------------------------------------------------------
def decode(ids, errors="replace"):

    text_bytes = b"".join(vocab[idx] for idx in ids)
    text = text_bytes.decode("utf-8", errors=errors)
    return text



metin = "Çığ yağdı, güneş açtı! 🌞🚀"

tokenlar = encode(metin)
geri = decode(tokenlar)

metin_bytes = metin.encode("utf-8")
print("metin            :", metin)
print("karakter sayısı  :", len(metin))
print("UTF-8 byte sayısı:", len(metin_bytes))
print("token sayısı     :", len(tokenlar))
print(f"compression ratio: {len(metin_bytes) / len(tokenlar):.2f}x")
print("tokenlar         :", tokenlar)

# Her token'ın temsil ettiği parça (emoji byte'ları gibi yarım karakterler '�' görünür)
print("token parçaları  :", [vocab[t].decode("utf-8", errors="replace") for t in tokenlar])
print("🌞 byte'ları      :", list("🌞".encode("utf-8")))
print("🚀 byte'ları      :", list("🚀".encode("utf-8")))

print("decode(tokenlar) :", geri)
print("decode(encode(metin)) == metin ->", geri == metin)





(196, 177) -> 256  'ı'
(196, 159) -> 257  'ğ'
(44, 32) -> 258  ', '
(197, 159) -> 259  'ş'
(101, 32) -> 260  'e '
(97, 114) -> 261  'ar'
(195, 188) -> 262  'ü'
(101, 114) -> 263  'er'
(97, 257) -> 264  'ağ'
(195, 167) -> 265  'ç'
(105, 110) -> 266  'in'
(108, 263) -> 267  'ler'
(32, 121) -> 268  ' y'
(101, 259) -> 269  'eş'
(256, 258) -> 270  'ı, '
metin            : Çığ yağdı, güneş açtı! 🌞🚀
karakter sayısı  : 25
UTF-8 byte sayısı: 40
token sayısı     : 21
compression ratio: 1.90x
tokenlar         : [195, 135, 294, 268, 264, 100, 270, 290, 291, 116, 256, 33, 32, 240, 159, 140, 158, 240, 159, 154, 128]
token parçaları  : ['�', '�', 'ığ', ' y', 'ağ', 'd', 'ı, ', 'güneş ', 'aç', 't', 'ı', '!', ' ', '�', '�', '�', '�', '�', '�', '�', '�']
🌞 byte'ları      : [240, 159, 140, 158]
🚀 byte'ları      : [240, 159, 154, 128]
decode(tokenlar) : Çığ yağdı, güneş açtı! 🌞🚀
decode(encode(metin)) == metin -> True


# **Task 4**

In [4]:
%pip -q install regex tiktoken pandas

In [5]:
import regex
import tiktoken
import pandas as pd
from IPython.display import display

# cl100k_base resmi Regex deseni
PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}++|\p{N}{1,3}+| ?[^\s\p{L}\p{N}]++[\r\n]*+|\s++$|\s*[\r\n]|\s+(?!\S)|\s"""

def merge(ids, pair, new_id):
    out = []
    i = 0
    while i < len(ids):
        if i + 1 < len(ids) and (ids[i], ids[i+1]) == pair:
            out.append(new_id)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out

class BasicTokenizer:
    def __init__(self):
        self.merges = {}
        self.vocab = {i: bytes([i]) for i in range(256)}

    def chunks(self, text):
        return [text]

    def train(self, text, vocab_size=512):
        assert vocab_size >= 256
        self.merges = {}
        self.vocab = {i: bytes([i]) for i in range(256)}
        chunks = [list(s.encode('utf-8')) for s in self.chunks(text)]
        for new_id in range(256, vocab_size):
            counts = {}
            for ids in chunks:
                for pair in zip(ids, ids[1:]):
                    counts[pair] = counts.get(pair, 0) + 1
            if not counts:
                break
            pair = max(counts, key=counts.get)
            chunks = [merge(ids, pair, new_id) for ids in chunks]
            self.merges[pair] = new_id
            self.vocab[new_id] = self.vocab[pair[0]] + self.vocab[pair[1]]
        return self

    def encode(self, text):
        output = []
        for chunk in self.chunks(text):
            ids = list(chunk.encode('utf-8'))
            while len(ids) > 1:
                pairs = set(zip(ids, ids[1:]))
                known = [pair for pair in pairs if pair in self.merges]
                if not known:
                    break
                pair = min(known, key=self.merges.get)  # eğitimde ilk öğrenilen
                ids = merge(ids, pair, self.merges[pair])
            output.extend(ids)
        return output

    def decode(self, ids):
        return b''.join(self.vocab[i] for i in ids).decode('utf-8')

class RegexTokenizer(BasicTokenizer):
    def chunks(self, text):
        return regex.findall(PATTERN, text)

def readable(token_bytes):
    return token_bytes.decode('utf-8', errors='backslashreplace').replace(' ', '␣')

In [6]:
TRAIN_TEXT = """
Sabah erken saatlerde yağmur yağdı. Yağmur dinince güneş açtı; sokaklar canlandı.
Öğrenciler kütüphanede kitap okudu, ders çalıştı ve arkadaşlarıyla projeler geliştirdi.
İstanbul'da vapura binen insanlar Kadıköy'den Beşiktaş'a geçiyor. Ankara'da otobüsler kalabalık.
Türkçe kelimelere ekler gelir: ev, evler, evlerimiz, evlerimizden; kitap, kitaplarımızdan.
Yapay zeka modelleri metinleri token olarak işler. Tokenizer harfleri, sayıları ve işaretleri ayırır.
İzmir'de yaz sıcak geçer, kışın yağmur yağar. Çocuklar bahçede oyun oynar, aileler çay içer.
""".strip() + "\n"
TRAIN_TEXT *= 12  # Küçük, tekrarlı bir eğitim korpusu; gerçek çalışma için büyütülebilir.

VOCAB_SIZE = 512
basic = BasicTokenizer().train(TRAIN_TEXT, VOCAB_SIZE)
custom = RegexTokenizer().train(TRAIN_TEXT, VOCAB_SIZE)

# Token ID'leri iki farklı tokenizer arasında eşdeğer değildir: byte içeriklerini karşılaştırıyoruz.
rows = []
basic_merges = list(basic.merges.items())
regex_merges = list(custom.merges.items())
for i in range(min(20, len(basic_merges), len(regex_merges))):
    (bp, bi), (rp, ri) = basic_merges[i], regex_merges[i]
    btext = f"{readable(basic.vocab[bp[0]])} + {readable(basic.vocab[bp[1]])} → {readable(basic.vocab[bi])}"
    rtext = f"{readable(custom.vocab[rp[0]])} + {readable(custom.vocab[rp[1]])} → {readable(custom.vocab[ri])}"
    rows.append({'Sıra': i + 1, 'Basic merge': btext, 'Regex merge': rtext})
display(pd.DataFrame(rows))

basic_tokens = {basic.vocab[i] for i in basic.merges.values()}
regex_tokens = {custom.vocab[i] for i in custom.merges.values()}
print('Merge sayısı (Basic / Regex):', len(basic.merges), len(custom.merges))
print('Ortak öğrenilen token:', len(basic_tokens & regex_tokens))
print('Sadece Basic / Sadece Regex:', len(basic_tokens - regex_tokens), '/', len(regex_tokens - basic_tokens))
TEST = "Kütüphanedeki öğrenciler Eskişehir'den İstanbul'a yeni bir proje için geldiler."
print('Test token sayısı (Basic / Regex):', len(basic.encode(TEST)), len(custom.encode(TEST)))
assert basic.decode(basic.encode(TEST)) == TEST
assert custom.decode(custom.encode(TEST)) == TEST

,Sıra,Basic merge,Regex merge
0,1,e + r → er,e + r → er
1,2,\xc4 + \xb1 → ı,\xc4 + \xb1 → ı
2,3,l + er → ler,l + er → ler
3,4,a + r → ar,a + r → ar
4,5,e + ␣ → e␣,\xc5 + \x9f → ş
5,6,\xc5 + \x9f → ş,\xc3 + \xa7 → ç
6,7,\xc3 + \xa7 → ç,e + n → en
7,8,", + ␣ → ,␣",l + ar → lar
8,9,e + n → en,␣ + k → ␣k
9,10,l + ar → lar,\xc4 + \x9f → ğ


Merge sayısı (Basic / Regex): 256 256
Ortak öğrenilen token: 35
Sadece Basic / Sadece Regex: 221 / 221
Test token sayısı (Basic / Regex): 63 51


In [7]:
words = ["Ankara'dan", "İstanbul'da", "Türkiye'nin", "İzmir'e",
         "Ankara’dan", "İstanbul’da", "evlerimizden"]
display(pd.DataFrame([
    {'Kelime': w, 'Regex parçaları': repr(regex.findall(PATTERN, w)),
     'Kendi BPE token': len(custom.encode(w))}
    for w in words
]))


,Kelime,Regex parçaları,Kendi BPE token
0,Ankara'dan,"['Ankara', ""'d"", 'an']",7
1,İstanbul'da,"['İstanbul', ""'d"", 'a']",3
2,Türkiye'nin,"['Türkiye', ""'nin""]",7
3,İzmir'e,"['İzmir', ""'e""]",3
4,Ankara’dan,"['Ankara', '’dan']",10
5,İstanbul’da,"['İstanbul', '’da']",6
6,evlerimizden,['evlerimizden'],7


# **Task 5**

In [8]:
TR = "Geçen hafta üniversitedeki arkadaşlarımla yapay zekâ destekli bir seyahat planlama uygulaması üzerine çalıştık. Uygulama, kullanıcıların bütçelerine, zamanlarına ve ilgi alanlarına göre farklı ulaşım seçeneklerini karşılaştırıyor. Örneğin İstanbul'dan Ankara'ya gitmek isteyen bir öğrenci, otobüs ve tren biletlerinin fiyatlarını tek ekranda görebiliyor. Sistem yalnızca en ucuz seçeneği önermekle kalmıyor; yolculuk süresini, aktarma sayısını ve tahmini karbon salımını da hesaba katıyor. Tasarımı test ederken kullanıcıların özellikle açık açıklamalara ve hızlı sonuçlara önem verdiğini fark ettik. Önümüzdeki ay daha fazla katılımcıyla yeni denemeler yapmayı ve elde ettiğimiz geri bildirimlere göre uygulamayı geliştirmeyi planlıyoruz."
EN = "Last week, I worked with my university friends on an artificial intelligence assisted travel planning app. The app compares different transportation options based on users' budgets, schedules, and interests. For example, a student who wants to travel from Istanbul to Ankara can see bus and train ticket prices on a single screen. The system does not simply recommend the cheapest option; it also considers travel time, the number of transfers, and estimated carbon emissions. While testing the design, we noticed that users particularly valued clear explanations and quick results. Next month, we plan to run new tests with more participants and improve the app based on the feedback we receive."

enc = tiktoken.get_encoding('cl100k_base')
assert enc._pat_str == PATTERN, "cl100k_base Regex deseni bu tiktoken sürümünde değişmiş."

def measure(language, tokenizer, text, ids, decode):
    return {'Dil': language, 'Tokenizer': tokenizer,
            'Kelime': len(text.split()), 'Token': len(ids),
            'Token/Kelime': round(len(ids) / len(text.split()), 3),
            'Round-trip doğru mu?': decode(ids) == text}

tr_ids = enc.encode(TR)
en_ids = enc.encode(EN)
my_ids = custom.encode(TR)

results = pd.DataFrame([
    measure('Türkçe', 'GPT-4 cl100k_base', TR, tr_ids, enc.decode),
    measure('İngilizce', 'GPT-4 cl100k_base', EN, en_ids, enc.decode),
    measure('Türkçe', 'Kendi RegexTokenizer (512)', TR, my_ids, custom.decode),
])
display(results)
print('GPT-4 toplam token oranı (Türkçe/İngilizce):', round(len(tr_ids)/len(en_ids), 3))
print('Türkçe token ID örneği (ilk 30):', tr_ids[:30])

,Dil,Tokenizer,Kelime,Token,Token/Kelime,Round-trip doğru mu?
0,Türkçe,GPT-4 cl100k_base,89,280,3.146,True
1,İngilizce,GPT-4 cl100k_base,110,126,1.145,True
2,Türkçe,Kendi RegexTokenizer (512),89,532,5.978,True


GPT-4 toplam token oranı (Türkçe/İngilizce): 2.222
Türkçe token ID örneği (ilk 30): [9688, 3209, 268, 6520, 728, 64, 10709, 77, 1986, 1639, 79853, 56317, 2649, 7370, 46937, 1029, 64, 40528, 352, 1167, 1247, 9011, 3281, 1247, 747, 15606, 513, 95780, 266, 3197]


In [9]:
pairs = [('ev', 'house'), ('evler', 'houses'),
         ('evlerimiz', 'our houses'), ('evlerimizden', 'from our houses')]
display(pd.DataFrame([
    {'Türkçe': tr, 'TR token (cl100k)': len(enc.encode(tr)),
     'İngilizce': en, 'EN token (cl100k)': len(enc.encode(en))}
    for tr, en in pairs
]))

,Türkçe,TR token (cl100k),İngilizce,EN token (cl100k)
0,ev,1,house,1
1,evler,2,houses,1
2,evlerimiz,4,our houses,2
3,evlerimizden,5,from our houses,3
